# Test API — Azure App Service

Hits the deployed Flask app on Azure. Same flow as the localhost test.

**Update `BASE_URL` below to match your App Service URL.**

**Before running:** make sure the latest `build_benefits.py` and `main.py` are deployed to App Service (otherwise you're testing the old code).


In [ ]:
import math
import json
import time
from pathlib import Path
import requests
import pandas as pd

# ── Config ────────────────────────────────────────────────────────────────────
BASE_URL      = 'https://desrapidev-fqaphqeaeuc7hafh.westcentralus-01.azurewebsites.net'
PAYLOAD_PATH  = './payloads/medicare_input_loadid142.json'
POLL_INTERVAL = 10    # seconds between polls
MAX_WAIT      = 600   # stop after 10 minutes
POST_TIMEOUT  = 120   # large payloads take time to upload + blob-write

# ── Helpers ───────────────────────────────────────────────────────────────────
def sanitize(obj):
    if isinstance(obj, dict):  return {k: sanitize(v) for k, v in obj.items()}
    if isinstance(obj, list):  return [sanitize(v) for v in obj]
    if isinstance(obj, float) and (math.isnan(obj) or math.isinf(obj)): return None
    return obj

# ── 1. Health check ───────────────────────────────────────────────────────────
print(f'Hitting {BASE_URL}')
try:
    r = requests.get(f'{BASE_URL}/', timeout=15)
    print(f'Health: {r.status_code} {r.json()}\n')
except requests.exceptions.ConnectionError as e:
    raise RuntimeError(f'Could not connect to {BASE_URL}.\nError: {e}')
except requests.exceptions.ReadTimeout:
    raise RuntimeError(
        f'Health check timed out hitting {BASE_URL}. '
        f'App Service may be cold-starting — wait 30s and try again.'
    )

# ── 2. Load + inspect payload (multi-plan diagnostic) ────────────────────────
raw  = json.loads(Path(PAYLOAD_PATH).read_text(encoding='utf-8'))
rows = raw['pbp'] if isinstance(raw, dict) and 'pbp' in raw else raw
clean = sanitize(rows)

# Count distinct plans BEFORE sending — this is the key diagnostic for
# the "only one plan got generated" complaint
filenames = {}
for r in clean:
    fn = r.get('FileName')
    if fn:
        filenames[fn] = filenames.get(fn, 0) + 1

payload_bytes = len(json.dumps(clean))
print(f'Payload: {len(clean):,} rows ({payload_bytes/1024/1024:.1f} MB)')
print(f'Distinct plans (FileNames) in input: {len(filenames)}')
for fn, n in sorted(filenames.items()):
    print(f'  - {fn}: {n:,} rows')
print()

# ── 3. POST payload ───────────────────────────────────────────────────────────
print(f'POSTing to {BASE_URL}/save ...')
t_post = time.monotonic()
try:
    r = requests.post(
        f'{BASE_URL}/save',
        json=clean,
        headers={'Content-Type': 'application/json'},
        timeout=POST_TIMEOUT,
    )
except requests.exceptions.ReadTimeout:
    raise RuntimeError(
        f'POST timed out after {POST_TIMEOUT}s. '
        f'Check App Service logs (az webapp log tail) for errors.'
    )
post_time = time.monotonic() - t_post

print(f'HTTP {r.status_code} in {post_time:.2f}s')
if r.status_code != 202:
    print(f'Unexpected status. Response body:')
    print(r.text)
    raise RuntimeError(f'Expected 202, got {r.status_code}')

resp = r.json()
print(json.dumps(resp, indent=2))

# Sanity check: server should report the same plan_count we saw client-side
server_plan_count = resp.get('plan_count')
if server_plan_count is not None and server_plan_count != len(filenames):
    print(f'\n⚠ Server saw {server_plan_count} plans but we sent {len(filenames)} — '
          f'something is stripping FileName upstream')

load_id   = resp['load_id']
blob_name = resp['blob_name']

# ── 4. Poll /results until done ───────────────────────────────────────────────
print(f'\nPolling /results/{load_id} every {POLL_INTERVAL}s...')
df = None
elapsed = 0
t_proc = time.monotonic()
while elapsed < MAX_WAIT:
    try:
        r = requests.get(f'{BASE_URL}/results/{load_id}', timeout=30)
    except requests.exceptions.ReadTimeout:
        print(f'  [{elapsed:>3}s] poll timed out, retrying...')
        time.sleep(POLL_INTERVAL)
        elapsed += POLL_INTERVAL
        continue

    data = r.json()
    status = data.get('status')
    print(f'  [{elapsed:>3}s] status={status}')

    if status == 'success':
        proc_time = time.monotonic() - t_proc
        print(f'\nDone in {proc_time:.1f}s')
        print(f'  Total rows:    {data.get("result_count")}')
        print(f'  Plan IDs out:  {data.get("plan_ids")}')

        # Validate the multi-plan complaint is fixed
        n_plans_out = data.get('plan_count', 0)
        if len(filenames) > 1 and n_plans_out < len(filenames):
            print(f'  ⚠ WARNING: input had {len(filenames)} plans, '
                  f'output only has {n_plans_out}')
        elif n_plans_out == len(filenames):
            print(f'  ✓ All {n_plans_out} input plans produced output')

        df = pd.DataFrame(data['results'])
        if not df.empty and {'benefitid','benefitname','serviceTypeDesc','tinyDescription'}.issubset(df.columns):
            print(f'\n  Rows per plan:')
            print(df.groupby('planid').size().to_string())
            display(df[['planid','benefitid','benefitname','serviceTypeDesc','tinyDescription']])
        else:
            print(f'\n  (no rows or unexpected schema, columns: {list(df.columns)})')
        break

    if status == 'error':
        print(f'\nERROR: {data.get("error")}')
        break

    time.sleep(POLL_INTERVAL)
    elapsed += POLL_INTERVAL
else:
    print('Timed out waiting for results.')

# ── 5. Save CSV ───────────────────────────────────────────────────────────────
if df is not None and not df.empty:
    CSV_OUT = f'output_benefits_loadid_{load_id}.csv'
    df.to_csv(CSV_OUT, index=False)
    print(f'\nSaved {len(df)} rows to {CSV_OUT}')
